# 05 — Fairness audit (per-identity FPR)

Loads cached `fairness_results.json` when present, or runs a quick live probe.
Full CLI: `python scripts/run_fairness.py`

In [ ]:
%matplotlib inline
import json, sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

_cwd = Path.cwd()
_root = _cwd.parent if _cwd.name == "notebooks" else _cwd
sys.path.insert(0, str(_root / "src"))

from paths import PROCESSED_DIR
from fairness import build_identity_probe_df, per_group_fpr
from model_registry import ALL_SPECS, load_scorer

RESULTS = PROCESSED_DIR / "fairness_results.json"
QUICK_LIVE = False  # set True to re-score probe live
MODELS = ["toxic-bert", "tfidf-svm"]  # add distilbert-student after Colab train
sns.set_theme(style="whitegrid")

In [ ]:
if RESULTS.exists() and not QUICK_LIVE:
    payload = json.loads(RESULTS.read_text(encoding="utf-8"))
    rows = []
    for key, block in payload["models"].items():
        rows.append({
            "model": key,
            "overall_fpr": block["overall_fpr"],
            "control_fpr": block["control_fpr"],
        })
    summary = pd.DataFrame(rows)
    display(summary.round(3))
    print(f"Loaded {RESULTS}")
else:
    from baselines import TfidfSvmScorer
    from splits import get_split_frame
    probe = build_identity_probe_df()
    rows = []
    for key in MODELS:
        spec = ALL_SPECS[key]
        scorer = load_scorer(spec)
        if key == "tfidf-svm":
            train = get_split_frame("train")
            scorer.fit(train["comment_text"].tolist(), train["toxic"].values)
        p = scorer.predict_toxic_proba(probe["text"].tolist())
        g = per_group_fpr(probe, p)
        rows.append({"model": key, "overall_fpr": g["fpr"].mean(), "control_fpr": float((p[probe["group"]=="control"]>=0.5).mean()) if (probe["group"]=="control").any() else float("nan")})
        display(g.sort_values("fpr", ascending=False).head())
    summary = pd.DataFrame(rows)
    display(summary.round(3))

In [ ]:
fig_path = _root / "report" / "figures" / "fairness_fpr_heatmap.png"
if fig_path.exists():
    from IPython.display import Image, display as idisplay
    idisplay(Image(filename=str(fig_path)))
else:
    print("Figure missing — run: python scripts/plot_fairness_robustness.py")